In [0]:
silver_df = spark.table("weather_silver")

display(silver_df)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

dim_location = (
    silver_df
    .select("city", "latitude", "longitude")
    .dropDuplicates(["city"])
    .orderBy("city")
    .withColumn(
        "location_id",
        F.row_number().over(Window.orderBy("city"))
    )
    .select(
        "location_id",
        "city",
        "latitude",
        "longitude"
    )
)

display(dim_location)

In [0]:
dim_date = (
    silver_df
    .select("date")
    .dropDuplicates()
    .withColumn("date_id", F.date_format("date", "yyyyMMdd").cast("int"))
    .withColumn("year", F.year("date"))
    .withColumn("month", F.month("date"))
    .withColumn("month_name", F.date_format("date", "MMMM"))
    .withColumn("day", F.dayofmonth("date"))
    .withColumn("day_of_week", F.dayofweek("date"))
    .select(
        "date_id",
        "date",
        "year",
        "month",
        "month_name",
        "day",
        "day_of_week"
    )
    .orderBy("date")
)

display(dim_date)

In [0]:
fact_weather = (
    silver_df.alias("s")
    .join(
        dim_location.alias("l"),
        F.col("s.city") == F.col("l.city"),
        "inner"
    )
    .join(
        dim_date.alias("d"),
        F.col("s.date") == F.col("d.date"),
        "inner"
    )
    .select(
        F.monotonically_increasing_id().alias("weather_id"),
        F.col("l.location_id"),
        F.col("d.date_id"),
        F.col("s.timestamp"),
        F.col("s.temperature_2m").alias("temperature"),
        F.col("s.relative_humidity_2m").alias("humidity"),
        F.col("s.precipitation"),
        F.col("s.wind_speed_10m").alias("wind_speed")
    )
)

display(fact_weather)

In [0]:
dim_location.write.format("delta").mode("overwrite").saveAsTable("dim_location")

dim_date.write.format("delta").mode("overwrite").saveAsTable("dim_date")

fact_weather.write.format("delta").mode("overwrite").saveAsTable("fact_weather")

In [0]:
display(spark.table("dim_location"))
display(spark.table("dim_date"))
display(spark.table("fact_weather"))